# Exercise 1: Getting Started with Neo4j - Building Your First Graph

Welcome to the world of graph databases! While not as common as relational and NoSQL databases, graph systems have their critical use cases in the world of data as well! In this exercise you'll explore graph databases through Neo4j, a powerful tool for storing and querying highly connected data. Rather than storing data as rows or documents, graphs store **nodes** (entities) connected by **relationships** (edges), enabling fast relationship traversals and pattern matching.

## Step 1: Connect to Neo4j and Initialize the Graph

First, we'll establish a connection to the Neo4j instance running in your workspace. Neo4j provides the Neo4j Python Driver for easy integration with Python and Jupyter notebooks.

In [ ]:
%%capture
# Install required packages
%pip install neo4j networkx matplotlib pandas openai

# Import required libraries
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable
import time
import pandas as pd
import json

# Driver setup - connect to Neo4j instance
URI = "neo4j://localhost:7687"
AUTH = ("neo4j", "password")

# Create driver
driver = GraphDatabase.driver(URI, auth=AUTH)

# Test connection
try:
    with driver.session() as session:
        session.run("RETURN 1")
    print("✅ Successfully connected to Neo4j")
except ServiceUnavailable:
    print("❌ Could not connect to Neo4j. Make sure the service is running.")
    driver.close()

## Step 2: Create Sample Data - Building Our Product Graph

Now we'll create a simple but realistic graph representing customers, products, purchases, and ratings. This graph will allow us to explore relationships and build recommendations based on purchase history and product similarity.

Let's start fresh by removing any existing nodes and relationships from previous runs.

In [ ]:
# Clear all existing nodes and relationships
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
    print("✅ Cleared all nodes and relationships")

### Creating Nodes

In Neo4j, we create nodes using the Cypher query language. Nodes can have labels (like 'Customer', 'Product') and properties (like name, email, price). Let's create our first nodes.

In [ ]:
# Create Customer nodes
create_customers_query = """
CREATE 
  (alice:Customer {id: 'C001', name: 'Alice Smith', email: 'alice@example.com'}),
  (bob:Customer {id: 'C002', name: 'Bob Johnson', email: 'bob@example.com'}),
  (charlie:Customer {id: 'C003', name: 'Charlie Lee', email: 'charlie@example.com'})
RETURN count(*) as CustomersCreated
"""

with driver.session() as session:
    result = session.run(create_customers_query)
    for record in result:
        print(f"✅ {record['CustomersCreated']} customers created")

In [ ]:
# Create Product nodes
create_products_query = """
CREATE
  (laptop:Product {id: 'P001', name: 'Laptop', category: 'Electronics', price: 999.99}),
  (mouse:Product {id: 'P002', name: 'Wireless Mouse', category: 'Accessories', price: 29.99}),
  (monitor:Product {id: 'P003', name: '4K Monitor', category: 'Electronics', price: 399.99}),
  (keyboard:Product {id: 'P004', name: 'Mechanical Keyboard', category: 'Accessories', price: 149.99}),
  (headphones:Product {id: 'P005', name: 'Wireless Headphones', category: 'Audio', price: 199.99})
RETURN count(*) as ProductsCreated
"""

with driver.session() as session:
    result = session.run(create_products_query)
    for record in result:
        print(f"✅ {record['ProductsCreated']} products created")

### Creating Relationships

Now we'll create relationships between customers and products to represent purchases and ratings. Note, th at purchases here are a *relationship* that connects two nodes - not a separate entity stored in a different table on their own! This is a unique difference between graph and relational/NoSQL databases.

In [ ]:
# Create PURCHASED relationships
create_purchases_query = """
MATCH (alice:Customer {id: 'C001'}), (laptop:Product {id: 'P001'})
CREATE (alice)-[:PURCHASED {date: '2024-01-15', quantity: 1}]->(laptop)

MATCH (alice:Customer {id: 'C001'}), (monitor:Product {id: 'P003'})
CREATE (alice)-[:PURCHASED {date: '2024-01-20', quantity: 1}]->(monitor)

MATCH (bob:Customer {id: 'C002'}), (laptop:Product {id: 'P001'})
CREATE (bob)-[:PURCHASED {date: '2024-02-01', quantity: 1}]->(laptop)

MATCH (bob:Customer {id: 'C002'}), (mouse:Product {id: 'P002'})
CREATE (bob)-[:PURCHASED {date: '2024-02-02', quantity: 2}]->(mouse)

MATCH (charlie:Customer {id: 'C003'}), (keyboard:Product {id: 'P004'})
CREATE (charlie)-[:PURCHASED {date: '2024-02-05', quantity: 1}]->(keyboard)

MATCH (charlie:Customer {id: 'C003'}), (headphones:Product {id: 'P005'})
CREATE (charlie)-[:PURCHASED {date: '2024-02-06', quantity: 1}]->(headphones)

RETURN 'Relationships created' as status
"""

with driver.session() as session:
    result = session.run(create_purchases_query)
    for record in result:
        print(f"✅ {record['status']}")

In [ ]:
# Create RATED relationships (product ratings)
create_ratings_query = """
MATCH (alice:Customer {id: 'C001'}), (laptop:Product {id: 'P001'})
CREATE (alice)-[:RATED {score: 5, review: 'Excellent laptop, very fast'}]->(laptop)

MATCH (alice:Customer {id: 'C001'}), (monitor:Product {id: 'P003'})
CREATE (alice)-[:RATED {score: 4, review: 'Great display quality'}]->(monitor)

MATCH (bob:Customer {id: 'C002'}), (laptop:Product {id: 'P001'})
CREATE (bob)-[:RATED {score: 4, review: 'Very reliable machine'}]->(laptop)

MATCH (bob:Customer {id: 'C002'}), (mouse:Product {id: 'P002'})
CREATE (bob)-[:RATED {score: 5, review: 'Perfect DPI settings'}]->(mouse)

RETURN 'Ratings created' as status
"""

with driver.session() as session:
    result = session.run(create_ratings_query)
    for record in result:
        print(f"✅ {record['status']}")

## Step 3: Visualize the Graph

Let's visualize the structure of our graph to understand the relationships better.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Query all nodes and relationships
query = """
MATCH (n)
OPTIONAL MATCH (n)-[r]-(m)
RETURN n, r, m
"""

# Create a NetworkX graph
G = nx.Graph()

with driver.session() as session:
    result = session.run(query)
    
    for record in result:
        node_n = record['n']
        node_m = record['m']
        rel = record['r']
        
        # Add nodes
        if node_n:
            node_id = node_n['id'] if 'id' in node_n else str(node_n)
            label = list(node_n.labels)[0] if node_n.labels else 'Node'
            G.add_node(node_id, label=label, name=node_n.get('name', node_id))
        
        if node_m and rel:  # Only add if there's a relationship
            node_id_m = node_m['id'] if 'id' in node_m else str(node_m)
            label_m = list(node_m.labels)[0] if node_m.labels else 'Node'
            G.add_node(node_id_m, label=label_m, name=node_m.get('name', node_id_m))
            
            # Add edge with relationship type
            rel_type = rel.type
            G.add_edge(node_id, node_id_m, relationship=rel_type)

# Visualize
plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

# Draw nodes by type
customer_nodes = [node for node, attr in G.nodes(data=True) if attr.get('label') == 'Customer']
product_nodes = [node for node, attr in G.nodes(data=True) if attr.get('label') == 'Product']

nx.draw_networkx_nodes(G, pos, nodelist=customer_nodes, node_color='lightblue', 
                       node_size=2000, label='Customers')
nx.draw_networkx_nodes(G, pos, nodelist=product_nodes, node_color='lightgreen',
                       node_size=2000, label='Products')

# Draw edges
nx.draw_networkx_edges(G, pos, width=2, alpha=0.6)

# Draw labels
labels = {node: G.nodes[node].get('name', node) for node in G.nodes()}
nx.draw_networkx_labels(G, pos, labels, font_size=10, font_weight='bold')

plt.title('E-Commerce Graph: Customers, Products, and Relationships', fontsize=16, fontweight='bold')
plt.legend(scatterpoints=1)
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"✅ Graph visualization complete")
print(f"   Nodes: {len(G.nodes())}")
print(f"   Edges: {len(G.edges())}")

## Step 4: Query the Graph with Cypher

Cypher is Neo4j's query language. It's designed for expressing graph patterns. It is similar to GQL, the new Graph Query Language standard that is now an ISO standard! It also visually closely aligns with how we would picture a graph as a data structure.

Let's write some queries to extract insights from our graph. 

#### Task 1: Find all products purchased by a specific customer

Let's write a query to find all products that customer Alice (C001) has purchased.

In [ ]:
# Find all products purchased by Alice
alice_purchases_query = """
### BEGIN SOLUTION
MATCH (customer:Customer {id: 'C001'})-[r:PURCHASED]->(product:Product)
RETURN customer.name as Customer, product.name as Product, r.date as PurchaseDate, r.quantity as Quantity
### END SOLUTION
"""

with driver.session() as session:
    result = session.run(alice_purchases_query)
    records = list(result)
    df = pd.DataFrame([record.values() for record in records], columns=records[0].keys() if records else [])
    print("Alice's Purchases:")
    print(df)
    print(f"\n✅ Found {len(records)} purchases")

#### Task 2: Find customers who have purchased the same product

Great work! Let's keep going. We don't only have to consider the *customer*. The product is also a node, and it has a relationship with *customers* as well. So, we can actually very easily use the graph to find all customers who purchased X or Y product. 

In this case, let's find all customers who purchased the laptop (P001).

In [ ]:
# Find all customers who purchased a specific product
laptop_customers_query = """
### BEGIN SOLUTION
MATCH (customer:Customer)-[r:PURCHASED]->(product:Product {id: 'P001'})
RETURN customer.name as Customer, product.name as Product, r.date as PurchaseDate
### END SOLUTION
"""

with driver.session() as session:
    result = session.run(laptop_customers_query)
    records = list(result)
    df = pd.DataFrame([record.values() for record in records], columns=records[0].keys() if records else [])
    print("Customers who purchased the Laptop:")
    print(df)
    print(f"\n✅ Found {len(records)} customers")

#### Task 3: Find products rated 5 stars

What else exists as a node, with relationships? Reviews! Let's run a similar query, but looking for entities that have relationships to a certain level of review. In this case, we'll try to find all products that have received a 5-star rating.

In [ ]:
# Find highly rated products
top_rated_query = """
### BEGIN SOLUTION
MATCH (customer:Customer)-[r:RATED {score: 5}]->(product:Product)
RETURN product.name as Product, customer.name as Reviewer, r.review as Review
### END SOLUTION
"""

with driver.session() as session:
    result = session.run(top_rated_query)
    records = list(result)
    df = pd.DataFrame([record.values() for record in records], columns=records[0].keys() if records else [])
    print("5-Star Rated Products:")
    print(df)
    print(f"\n✅ Found {len(records)} highly rated products")

## Step 5: Advanced Queries - Relationships Between Relationships

We don't *only* have to query in a pattern where we are looking for "nodes XYZ with direct connections to nodes ABC". One of the powerful features of graph databases is querying patterns across multiple hops in the graph. For example, using the syntax below with a where other_customer.id != ?????, where the placeholder is Alice's ID - we can actually traverse the graph:

1. First we start with Alice, and what she has purchased
2. Then we look at the products (nodes) she's purchased, and in turn,
3. We look at the relationships those products have with *other* customers. 

#### Task 4: Find customers with similar purchase history

Find customers who purchased the same products as Alice (multi-hop relationship).

In [ ]:
# Find customers with overlapping purchases
similar_customers_query = """
### BEGIN SOLUTION
MATCH (alice:Customer {id: 'C001'})-[:PURCHASED]->(product:Product)<-[:PURCHASED]-(other_customer:Customer)
### BEGIN SOLUTION
WHERE other_customer.id != 'C001'
### END SOLUTION
RETURN DISTINCT other_customer.name as SimilarCustomer, product.name as SharedProduct
"""

with driver.session() as session:
    result = session.run(similar_customers_query)
    records = list(result)
    df = pd.DataFrame([record.values() for record in records], columns=records[0].keys() if records else [])
    print("Customers with Similar Purchases to Alice:")
    print(df)
    print(f"\n✅ Found {len(records)} connections")

## Step 6: Build a Recommendation Engine using Graph

It might not seem like it - but we almost have all the steps we need to create a very simple data-driven workflow. Now we'll create a simple recommendation system that uses the graph structure to find candidate products, and then uses a simple system to rank and refine recommendations. 

### Part 1: Graph-Based Product Retrieval

First, we'll retrieve candidate products using graph queries. The idea is to find products that:
1. Have been purchased by customers similar to our target customer
2. Haven't been purchased by the target customer yet
3. Have high ratings

In [ ]:
# Function to get product recommendations using graph traversal
def get_recommendations_from_graph(customer_id: str) -> list:
    """
    Get product recommendations using graph-based retrieval.
    Strategy: Find products purchased by customers with similar tastes.
    """
    recommendation_query = f"""
    MATCH (target_customer:Customer {{id: '{customer_id}'}})-[:PURCHASED]->(target_products:Product)
    MATCH (similar_customer:Customer)-[:PURCHASED]->(target_products)
    WHERE similar_customer.id != target_customer.id
    MATCH (similar_customer)-[:PURCHASED]->(candidate_product:Product)
    WHERE NOT (target_customer)-[:PURCHASED]->(candidate_product)
    OPTIONAL MATCH (rater:Customer)-[rating:RATED {{score: 5}}]->(candidate_product)
    RETURN 
        candidate_product.id as product_id,
        candidate_product.name as product_name,
        candidate_product.category as category,
        candidate_product.price as price,
        COUNT(DISTINCT rater) as five_star_ratings,
        COUNT(DISTINCT similar_customer) as similar_customers_count
    ORDER BY five_star_ratings DESC, similar_customers_count DESC
    LIMIT 5
    """
    
    recommendations = []
    with driver.session() as session:
        result = session.run(recommendation_query)
        for record in result:
            recommendations.append({
                'product_id': record['product_id'],
                'product_name': record['product_name'],
                'category': record['category'],
                'price': record['price'],
                'five_star_ratings': record['five_star_ratings'],
                'similar_customers_count': record['similar_customers_count']
            })
    return recommendations

# Get recommendations for Bob
bob_recommendations = get_recommendations_from_graph('C002')
print("\n🎯 Recommendation Candidates for Bob (from graph):")
for i, rec in enumerate(bob_recommendations, 1):
    print(f"\n  {i}. {rec['product_name']}")
    print(f"     Category: {rec['category']}")
    print(f"     Price: ${rec['price']:.2f}")
    print(f"     5-Star Ratings: {rec['five_star_ratings']}")
    print(f"     Customers with Similar Interests: {rec['similar_customers_count']}")

### Part 2: Graph-Powered Recommendation Refinement

Now we'll use a scoring algorithm to personalize and rank these recommendations based on semantic similarity and customer preferences. In practice, you can also use AI here, but for the simplicity of this exercise we have written our own ranking method.

In [ ]:
# Simulated AI recommendation ranker
# In a real implementation, this would use embeddings and LLMs

def ai_rank_recommendations(customer_id: str, recommendations: list, customer_profile: dict) -> list:
    """
    Use AI logic to rank and personalize recommendations.
    In production, you would use:
    - OpenAI embeddings for semantic similarity
    - Retrieved context from the graph
    - LLM to generate explanations
    """
    ranked_recommendations = []
    
    for rec in recommendations:
        # Simple AI ranking logic (in production, use embeddings/LLMs)
        # Score based on multiple factors
        score = 0
        reason = []
        
        # Factor 1: Quality (5-star ratings)
        if rec['five_star_ratings'] > 0:
            score += rec['five_star_ratings'] * 20
            reason.append(f"Highly rated ({rec['five_star_ratings']} 5-star reviews)")
        
        # Factor 2: Peer interest (similar customers interested)
        if rec['similar_customers_count'] > 0:
            score += rec['similar_customers_count'] * 15
            reason.append(f"{rec['similar_customers_count']} similar customers like it")
        
        # Factor 3: Category preference (if customer likes electronics, boost electronics)
        if customer_profile.get('preferred_category') == rec['category']:
            score += 10
            reason.append(f"Matches your interest in {rec['category']}")
        
        ranked_recommendations.append({
            **rec,
            'ai_score': score,
            'recommendation_reason': ' • '.join(reason) if reason else 'Popular choice'
        })
    
    # Sort by AI score
    ranked_recommendations.sort(key=lambda x: x['ai_score'], reverse=True)
    return ranked_recommendations

# Get Bob's profile (could be retrieved from database)
bob_profile = {
    'customer_id': 'C002',
    'preferred_category': 'Accessories'
}

# Rank recommendations using AI logic
ranked_recommendations = ai_rank_recommendations('C002', bob_recommendations, bob_profile)

print("\n🤖 AI-Ranked Recommendations for Bob:")
for i, rec in enumerate(ranked_recommendations, 1):
    print(f"\n  {i}. {rec['product_name']} (Score: {rec['ai_score']})")
    print(f"     Category: {rec['category']} | Price: ${rec['price']:.2f}")
    print(f"     Why: {rec['recommendation_reason']}")

### Part 3: Create a Recommendation Chatbot

Let's create a simple recommendation chatbot that uses the graph + recommendation pipeline.

In [ ]:
class RecommendationChatbot:
    """
    A simple recommendation chatbot that uses graph + AI retrieval.
    This demonstrates the RAG (Retrieval Augmented Generation) pattern.
    """
    
    def __init__(self, driver):
        self.driver = driver
    
    def get_customer_info(self, customer_id: str) -> dict:
        """Retrieve customer information from graph"""
        query = f"""
        MATCH (customer:Customer {{id: '{customer_id}'}})
        OPTIONAL MATCH (customer)-[:PURCHASED]->(product:Product)
        RETURN 
            customer.name as name,
            customer.email as email,
            COUNT(DISTINCT product) as products_purchased
        """
        
        with self.driver.session() as session:
            result = session.run(query)
            record = result.single()
            if record:
                return {
                    'customer_id': customer_id,
                    'name': record['name'],
                    'email': record['email'],
                    'purchase_count': record['products_purchased']
                }
        return None
    
    def get_purchase_history(self, customer_id: str) -> list:
        """Retrieve customer's purchase history from graph"""
        query = f"""
        MATCH (customer:Customer {{id: '{customer_id}'}})-[r:PURCHASED]->(product:Product)
        RETURN product.name as product_name, r.date as purchase_date, r.quantity as quantity
        ORDER BY r.date DESC
        """
        
        purchases = []
        with self.driver.session() as session:
            result = session.run(query)
            for record in result:
                purchases.append({
                    'product': record['product_name'],
                    'date': record['purchase_date'],
                    'quantity': record['quantity']
                })
        return purchases
    
    def generate_recommendation_message(self, customer_id: str) -> str:
        """Generate a personalized recommendation message using graph + AI"""
        # Step 1: Get customer info (Graph Retrieval)
        customer_info = self.get_customer_info(customer_id)
        if not customer_info:
            return f"Customer {customer_id} not found."
        
        # Step 2: Get purchase history (Graph Retrieval)
        purchase_history = self.get_purchase_history(customer_id)
        
        # Step 3: Get recommendations using graph + AI
        recommendations = get_recommendations_from_graph(customer_id)
        
        # Step 4: Use AI to rank recommendations
        customer_profile = {'customer_id': customer_id, 'preferred_category': None}
        ranked_recs = ai_rank_recommendations(customer_id, recommendations, customer_profile)
        
        # Step 5: Generate message
        message = f"""
Hello {customer_info['name']}! 👋

Based on your purchase history and customers similar to you, 
we have some great recommendations:
"""
        
        if ranked_recs:
            for i, rec in enumerate(ranked_recs[:3], 1):
                message += f"\n\n{i}. {rec['product_name']} (${rec['price']:.2f})"
                message += f"\n   Why: {rec['recommendation_reason']}"
        else:
            message += "\n\nNo new recommendations at this time. Check back soon!"
        
        return message

# Create chatbot instance
chatbot = RecommendationChatbot(driver)

# Generate recommendation for Alice
alice_message = chatbot.generate_recommendation_message('C001')
print(alice_message)

print("\n" + "="*60 + "\n")

# Generate recommendation for Charlie
charlie_message = chatbot.generate_recommendation_message('C003')
print(charlie_message)

## Summary: Key Takeaways

Great work getting through the material. In this exercise, we've explored the power of graph databases through Neo4j. Keep these 4 key points in mind when working with graphs in the future!

1. **Node-Relationship Model**: Graph databases store entities as nodes and their connections as relationships, enabling efficient traversal.

2. **Pattern Matching**: Cypher enables powerful pattern matching queries (finding customers with similar interests in just a few lines).

3. **Relationship Traversal**: Unlike relational databases that require multiple JOINs, graphs traverse relationships in O(1) time.

4. **Common Use Cases** - Graphs are great at things like:
   - Recommendation engines
   - Social networks
   - Knowledge graphs
   - Fraud detection (by finding unusual patterns)
   - Identity and access management